In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, TargetEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


In [4]:

df = pd.read_pickle('/Users/komarevtsev/Documents/project_Komarevtsev/data/clean_data.pkl')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1152 entries, 3 to 1999
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   battery_power       1152 non-null   int16   
 1   blue                1152 non-null   category
 2   clock_speed         1152 non-null   float32 
 3   dual_sim            1152 non-null   category
 4   fc                  1152 non-null   int8    
 5   four_g              1152 non-null   category
 6   int_memory          1152 non-null   int8    
 7   m_dep               1152 non-null   float32 
 8   mobile_wt           1152 non-null   int16   
 9   n_cores             1152 non-null   category
 10  pc                  1152 non-null   int8    
 11  px_height           1152 non-null   int16   
 12  px_width            1152 non-null   int16   
 13  ram                 1152 non-null   int16   
 14  sc_h                1152 non-null   int8    
 15  sc_w                1152 non-null   int8   

In [6]:
X = df.drop(columns=['price_range'])
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Определяем типы признаков
num_features = X_train.select_dtypes(include=['number']).columns.tolist()
cat_features = X_train.select_dtypes(include=['category', 'object', 'bool']).columns.tolist()

print(f"Числовые признаки ({len(num_features)}): {num_features}")
print(f"Категориальные признаки ({len(cat_features)}): {cat_features}")

Числовые признаки (17): ['battery_power', 'clock_speed', 'fc', 'int_memory', 'm_dep', 'mobile_wt', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'total_pixels', 'pixel_density', 'battery_efficiency', 'total_camera']
Категориальные признаки (7): ['blue', 'dual_sim', 'four_g', 'n_cores', 'three_g', 'touch_screen', 'wifi']


In [7]:

# Базовый препроцессор
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features)
    ],
    remainder='drop'
)

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', baseline_preprocessor),
    ('model', RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)) 
])

# Обучение модели
baseline_pipeline.fit(X_train, y_train)

# Предсказания
y_pred = baseline_pipeline.predict(X_test)
y_pred_proba = baseline_pipeline.predict_proba(X_test)

# Метрики качества  (precision, recall, f1, roc_auc)

baseline_metrics = {
    'precision': precision_score(y_test, y_pred, average='macro'),
    'recall': recall_score(y_test, y_pred, average='macro'),
    'f1': f1_score(y_test, y_pred, average='macro'),
    'roc_auc': roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
}

print("Baseline metrics:", baseline_metrics)

Baseline metrics: {'precision': 0.8506346104106143, 'recall': 0.8552067495681233, 'f1': 0.851730050126597, 'roc_auc': 0.9628980842506698}


In [8]:

# Подключаемся к локальному MLflow серверу 
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

tracking_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)

print(f" MLflow tracking URI: {mlflow.get_tracking_uri()}")

 MLflow tracking URI: http://127.0.0.1:5000


In [11]:
# Название эксперимента и модели
EXPERIMENT_NAME = "mobile_price_research"
RUN_NAME = "baseline_random_forest"
REGISTRY_MODEL_NAME = "mobile_price_rf"

# эксперимент
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id

# Готовим артефакты
input_example = X_train.head(5)  # Пример входных данных
signature = infer_signature(model_input=X_train.head(5))  # Сигнатура модели



/Users/komarevtsev/Documents/project_Komarevtsev/.venv_project_Komarevtsev/lib/python3.12/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [15]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    #  Логируем саму модель (вместе с препроцессором)
    mlflow.sklearn.log_model(
        sk_model=baseline_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        pip_requirements=["scikit-learn", "pandas", "numpy"]
    )
    
    #  Логируем метрики
    mlflow.log_metrics(baseline_metrics)
    
    #  Логируем параметры модели
    mlflow.log_params(baseline_pipeline.get_params())
    




2026/09/29 03:21:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/fca67c2bd15649c8911719db36a19a77.
2026/09/29 03:21:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [17]:
# Регистрируем модель как версию 1
model_version = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=REGISTRY_MODEL_NAME
)



Successfully registered model 'mobile_price_rf'.
2026/09/29 03:24:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_rf, version 1
Created version '1' of model 'mobile_price_rf'.


### Feature Engineering с помощью sklearn

In [17]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer

# Создаём копию обучающей выборки 
X_train_fe_sklearn = X_train.copy()

print(f"Исходный размер X_train: {X_train_fe_sklearn.shape}")

Исходный размер X_train: (864, 24)


In [18]:
# PolynomialFeatures для 2 признаков (ram и battery_power )
poly_features = ['ram', 'battery_power']
poly_pipeline = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('scale', StandardScaler())  
])

# KBinsDiscretizer для 3 признаков 
bin_features = ['mobile_wt', 'sc_h', 'talk_time']
binner = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')

In [19]:
# Новый ColumnTransformer = базовые трансформации + новые
fe_preprocessor = ColumnTransformer(
    transformers=[
        # Базовые трансформации (как в baseline)
        ('num', StandardScaler(), num_features),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features),
        
        # Новые трансформации
        ('poly', poly_pipeline, poly_features),
        ('binner', binner, bin_features)
    ],
    remainder='drop'
)



In [20]:
# Обучаем трансформер и получаем преобразованные данные
X_train_fe_sklearn = fe_preprocessor.fit_transform(X_train_fe_sklearn)

# Получаем имена всех новых признаков
new_feature_names = fe_preprocessor.get_feature_names_out()

print(f"Количество признаков после FE: {len(new_feature_names)}")
print(f"Было: {len(num_features) + len(cat_features)}")
print(f"Стало: {len(new_feature_names)}")
print(f"\nПервые 10 новых признаков:")
for name in new_feature_names[:10]:
    print(f"  - {name}")

# Сохраняем названия столбцов в файл 
with open('fe_columns.txt', 'w') as f:
    f.write("\n".join(new_feature_names))



Количество признаков после FE: 30
Было: 24
Стало: 30

Первые 10 новых признаков:
  - num__battery_power
  - num__clock_speed
  - num__fc
  - num__int_memory
  - num__m_dep
  - num__mobile_wt
  - num__pc
  - num__px_height
  - num__px_width
  - num__ram


In [21]:
# Создаём pipeline: сначала трансформации, потом модель
fe_pipeline = Pipeline(steps=[
    ('preprocessor', fe_preprocessor),
    ('model', RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42))
])



In [22]:
# Обучаем модель
fe_pipeline.fit(X_train, y_train)

# Делаем предсказания на тестовой выборке
y_pred_fe = fe_pipeline.predict(X_test)
y_pred_proba_fe = fe_pipeline.predict_proba(X_test)

# Вычисляем метрики
fe_metrics = {
    'precision': precision_score(y_test, y_pred_fe, average='macro', zero_division=0),
    'recall': recall_score(y_test, y_pred_fe, average='macro', zero_division=0),
    'f1': f1_score(y_test, y_pred_fe, average='macro', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_fe, multi_class='ovr', average='macro')
}

print("Метрики FE-модели:")
for metric, value in fe_metrics.items():
    print(f"  {metric}: {value:.4f}")



Метрики FE-модели:
  precision: 0.8886
  recall: 0.8898
  f1: 0.8889
  roc_auc: 0.9841


In [30]:
# Название запуска 
RUN_NAME_FE = "feature_engineering_sklearn"

with mlflow.start_run(run_name=RUN_NAME_FE, experiment_id=experiment_id) as run:
    fe_run_id = run.info.run_id
    
    # Логируем модель
    mlflow.sklearn.log_model(
        sk_model=fe_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        pip_requirements=['scikit-learn', 'pandas', 'numpy']
    )
    
    # Логируем метрики
    mlflow.log_metrics(fe_metrics)
    
    # Логируем параметры модели
    mlflow.log_params(fe_pipeline.get_params())
    
    # Логируем файл с названиями столбцов (артефакт!)
    mlflow.log_artifact('fe_columns.txt')
    
  

2026/09/29 03:44:13 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_engineering_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/aab7f718738e4eab9aecf3f840bb23ac.
2026/09/29 03:44:13 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


### AutoFeat


In [12]:
from autofeat import AutoFeatClassifier
import numpy as np
import pandas as pd

# Применяем AutoFeat ОТДЕЛЬНО (как в примере преподавателя)
# Не через ColumnTransformer!
afclf = AutoFeatClassifier(
    verbose=1, 
    feateng_steps=2, 
    max_gb=8, 
    transformations=["log", "sqrt"],
    feateng_cols=num_features
)

# Применяем только к числовым признакам
X_train_num_af = afclf.fit_transform(X_train[num_features], y_train)
X_test_num_af = afclf.transform(X_test[num_features])

print(f"Исходно числовых признаков: {len(num_features)}")
print(f"После AutoFeat: {X_train_num_af.shape[1]}")
print("\nНовые признаки:")
display(X_train_num_af.head())

2026-09-29 04:41:43,578 INFO: [AutoFeat] The 2 step feature engineering process could generate up to 595 features.
2026-09-29 04:41:43,579 INFO: [AutoFeat] With 864 data points this new feature matrix would use about 0.00 gb of space.
2026-09-29 04:41:43,580 INFO: [feateng] Step 1: transformation of original features


2026-09-29 04:41:44,079 INFO: [feateng] Generated 31 transformed features from 17 original features - done.
2026-09-29 04:41:44,080 INFO: [feateng] Step 2: first combination of features
2026-09-29 04:41:44,277 INFO: [feateng] Generated 1128 feature combinations from 1128 original feature tuples - done.
2026-09-29 04:41:44,279 INFO: [feateng] Generated altogether 1162 new features in 2 steps
2026-09-29 04:41:44,279 INFO: [feateng] Removing correlated features, as well as additions at the highest level
2026-09-29 04:41:44,289 INFO: [feateng] Generated a total of 767 additional features
2026-09-29 04:41:44,302 INFO: [featsel] Feature selection run 1/5


[featsel] Scaling data...done.       1128 feature tuples combined


2026-09-29 04:42:41,878 INFO: [featsel] Feature selection run 2/5
2026-09-29 04:43:38,974 INFO: [featsel] Feature selection run 3/5
2026-09-29 04:44:35,685 INFO: [featsel] Feature selection run 4/5
2026-09-29 04:45:31,320 INFO: [featsel] Feature selection run 5/5
2026-09-29 04:46:27,350 INFO: [featsel] 30 features after 5 feature selection runs
/Users/komarevtsev/Documents/project_Komarevtsev/.venv_project_Komarevtsev/lib/python3.12/site-packages/autofeat/featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
2026-09-29 04:46:27,353 INFO: [featsel] 13 features after correlation filtering
2026-09-29 04:46:28,565 INFO: [featsel] 12 features after noise filtering
2026-09-29 04:46:28,565 INFO: [AutoFeat] Computing 11 new features.


2026-09-29 04:46:29,037 INFO: [AutoFeat]    11/   11 new features ...done.
2026-09-29 04:46:29,038 INFO: [AutoFeat] Final dataframe with 28 feature columns (11 new).
2026-09-29 04:46:29,038 INFO: [AutoFeat] Training final classification model.


2026-09-29 04:46:29,359 INFO: [AutoFeat] Trained model: largest coefficients:
2026-09-29 04:46:29,359 INFO: [ 1.06218762e-11  3.02746603e-12 -2.13822586e-12 -1.15111164e-11]
2026-09-29 04:46:29,360 INFO: [AutoFeat] Final score: 0.3449
2026-09-29 04:46:29,362 INFO: [AutoFeat] Computing 11 new features.
2026-09-29 04:46:29,365 INFO: [AutoFeat]    11/   11 new features ...done.


Исходно числовых признаков: 17tures
После AutoFeat: 28

Новые признаки:


,battery_power,clock_speed,fc,int_memory,m_dep,mobile_wt,pc,px_height,px_width,ram,...,px_width*ram,battery_power*ram,sqrt(px_height)*ram,mobile_wt*ram,sqrt(m_dep)*ram,battery_power*total_pixels,ram*log(int_memory),sqrt(pixel_density)*ram,sqrt(fc)*ram,pc*total_pixels
0,808.0,1.9,7.0,20.0,0.8,185.0,14.0,496.0,974.0,1471.0,...,1432754.0,1188568.0,32760.725511,272135.0,1315.702408,3.903480e+08,4406.722174,141785.336286,3891.900179,6763456.0
1,536.0,2.4,12.0,3.0,0.3,182.0,14.0,1386.0,1539.0,284.0,...,437076.0,152224.0,10573.041946,51688.0,155.553209,1.143317e+09,312.005890,39912.358086,983.804859,29862756.0
2,1733.0,2.8,3.0,36.0,1.0,119.0,20.0,357.0,527.0,3576.0,...,1884552.0,6197208.0,67566.530413,425544.0,3576.000000,3.260449e+08,12814.663724,94048.630038,6193.813688,3762780.0
3,1641.0,1.1,6.0,48.0,0.7,97.0,7.0,224.0,1994.0,1695.0,...,3379830.0,2781495.0,25368.437082,164415.0,1418.138733,7.329625e+08,6561.685713,128265.242729,4151.885114,3126592.0
4,712.0,0.5,6.0,27.0,0.5,86.0,11.0,1245.0,1309.0,2001.0,...,2619309.0,1424712.0,70604.399615,172086.0,1414.920669,1.160350e+09,6594.969569,243559.596024,4901.428975,17926755.0


In [14]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Обрабатываем категориальные признаки отдельно
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_cat = cat_encoder.fit_transform(X_train[cat_features])
X_test_cat = cat_encoder.transform(X_test[cat_features])



In [15]:
from scipy.sparse import hstack

# Объединяем числовые (после AutoFeat) и категориальные признаки
X_train_all = np.hstack([X_train_num_af.values, X_train_cat])
X_test_all = np.hstack([X_test_num_af.values, X_test_cat])

# Шкалируем всё вместе
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_test_scaled = scaler.transform(X_test_all)

# Создаём имена колонок
num_names = X_train_num_af.columns.tolist()
cat_names = [f'cat__{c}' for c in cat_features]
all_names = num_names + cat_names

# Создаём DataFrame
X_train_final = pd.DataFrame(X_train_scaled, columns=all_names)
X_test_final = pd.DataFrame(X_test_scaled, columns=all_names)

print(f"Итоговое количество признаков: {len(all_names)}")
print(f"Имена признаков: {all_names[:10]}...")

Итоговое количество признаков: 35
Имена признаков: ['battery_power', 'clock_speed', 'fc', 'int_memory', 'm_dep', 'mobile_wt', 'pc', 'px_height', 'px_width', 'ram']...


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Создаём и обучаем модель
model_af = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
model_af.fit(X_train_final, y_train)

# Предсказания
y_pred_af = model_af.predict(X_test_final)
y_pred_proba_af = model_af.predict_proba(X_test_final)

# Метрики
metrics_af = {
    'precision': precision_score(y_test, y_pred_af, average='macro', zero_division=0),
    'recall': recall_score(y_test, y_pred_af, average='macro', zero_division=0),
    'f1': f1_score(y_test, y_pred_af, average='macro', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_af, multi_class='ovr', average='macro')
}

print(" AutoFeat metrics:")
for metric, value in metrics_af.items():
    print(f"  {metric}: {value:.4f}")

print(f"\n Сравнение:")
print(f"  Baseline f1: {baseline_metrics['f1']:.4f}")
print(f"  FE sklearn f1: {fe_metrics['f1']:.4f}")
print(f"  AutoFeat f1:   {metrics_af['f1']:.4f}")

 AutoFeat metrics:
  precision: 0.9340
  recall: 0.9348
  f1: 0.9342
  roc_auc: 0.9936

 Сравнение:
  Baseline f1: 0.8517
  FE sklearn f1: 0.8889
  AutoFeat f1:   0.9342


In [24]:
# Создаём обёртку для логирования (чтобы можно было использовать в pipeline)
class AutoFeatPipeline:
    def __init__(self, afclf, cat_encoder, scaler, num_features, cat_features):
        self.afclf = afclf
        self.cat_encoder = cat_encoder
        self.scaler = scaler
        self.num_features = num_features
        self.cat_features = cat_features
    
    def predict(self, X):
        X_num = self.afclf.transform(X[self.num_features])
        X_cat = self.cat_encoder.transform(X[self.cat_features])
        X_all = np.hstack([X_num.values, X_cat])
        X_scaled = self.scaler.transform(X_all)
        return model_af.predict(X_scaled)
    
    def predict_proba(self, X):
        X_num = self.afclf.transform(X[self.num_features])
        X_cat = self.cat_encoder.transform(X[self.cat_features])
        X_all = np.hstack([X_num.values, X_cat])
        X_scaled = self.scaler.transform(X_all)
        return model_af.predict_proba(X_scaled)

# Для логирования создаём простой pipeline
from sklearn.pipeline import Pipeline

# Сохраняем все компоненты
af_pipeline_for_mlflow = Pipeline(steps=[
    ('model', model_af)
])

# Логирование
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME_AF = 'autofeat'

with mlflow.start_run(run_name=RUN_NAME_AF, experiment_id=experiment_id) as run:
    run_id_af = run.info.run_id
    
    # Логируем модель (только финальную модель, так как препроцессинг делали вручную)
    mlflow.sklearn.log_model(
        af_pipeline_for_mlflow,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=['scikit-learn', 'pandas', 'numpy', 'autofeat']
    )
    
    # Логируем метрики
    mlflow.log_metrics(metrics_af)
    
    # Логируем параметры
    mlflow.log_params(model_af.get_params())
    
    # Сохраняем информацию о признаках
    with open('autofeat_features.txt', 'w') as f:
        f.write('\n'.join(all_names))
    mlflow.log_artifact('autofeat_features.txt')
    
    

/Users/komarevtsev/Documents/project_Komarevtsev/.venv_project_Komarevtsev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/29 04:48:54 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "battery_power",
      "blue",
      "clock_speed",
      "dual_sim",
      "fc",
      "four_g",
      "int_memory",
      "m_dep",
      "mobile_wt",
      "n_cores",
      "pc",
      "px_height",
      "px_width",
      "ram",
      "sc_h",
      "sc_w",
      "talk_time",
      "three_g",
      "touch_screen",
      "wifi",
      "total_pixels",
      "pixel_density",
      "battery_efficiency",
      "total_camera"
    ],
    "data": [
      [
        808,
        0,
        1.899999976158142,
        0,
        7,
        0,
        20,
   

### Экспертный отбор признаков (на основе EDA)

In [25]:
import numpy as np

# Посмотрим, сколько всего признаков после FE из пункта 10
print(f"Всего признаков после FE: {len(new_feature_names)}")
print("\nВсе признаки:")
for i, name in enumerate(new_feature_names):
    print(f"  [{i:2d}] {name}")

# Выбираем N = 50% от общего количества (в диапазоне 20-70%)
N = int(len(new_feature_names) * 0.5)
print(f"\nБудем отбирать N = {N} признаков (50% от общего числа)")

Всего признаков после FE: 30

Все признаки:
  [ 0] num__battery_power
  [ 1] num__clock_speed
  [ 2] num__fc
  [ 3] num__int_memory
  [ 4] num__m_dep
  [ 5] num__mobile_wt
  [ 6] num__pc
  [ 7] num__px_height
  [ 8] num__px_width
  [ 9] num__ram
  [10] num__sc_h
  [11] num__sc_w
  [12] num__talk_time
  [13] num__total_pixels
  [14] num__pixel_density
  [15] num__battery_efficiency
  [16] num__total_camera
  [17] cat__blue
  [18] cat__dual_sim
  [19] cat__four_g
  [20] cat__n_cores
  [21] cat__three_g
  [22] cat__touch_screen
  [23] cat__wifi
  [24] poly__ram
  [25] poly__battery_power
  [26] poly__ram battery_power
  [27] binner__mobile_wt
  [28] binner__sc_h
  [29] binner__talk_time

Будем отбирать N = 15 признаков (50% от общего числа)


In [26]:
# На основе EDA выбираем наиболее важные признаки:
# 1. ram и его производные (самый важный признак, корреляция ~0.92)
# 2. battery_power и его производные
# 3. Ключевые категориальные признаки (four_g, wifi, touch_screen)
# 4. Основные числовые признаки (px_height, px_width, sc_h, sc_w)

# Список имён признаков, которые оставляем
selected_feature_names = [
    # Исходные важные числовые признаки
    'num__ram',
    'num__battery_power',
    'num__px_height',
    'num__px_width',
    'num__sc_h',
    'num__sc_w',
    'num__mobile_wt',
    
    # Исходные важные категориальные признаки
    'cat__four_g',
    'cat__wifi',
    'cat__touch_screen',
    'cat__dual_sim',
    
    # Полиномиальные признаки (ram × battery_power и их квадраты)
    'poly__ram',
    'poly__battery_power',
    'poly__ram battery_power',
    
    # Бинированные признаки
    'binner__mobile_wt',
    'binner__sc_h',
]

# Проверяем, что все выбранные признаки есть в нашем наборе
available_names = [name for name in selected_feature_names if name in new_feature_names]
print(f"Выбрано признаков: {len(available_names)} из {len(new_feature_names)}")
print(f"Доля: {len(available_names)/len(new_feature_names)*100:.1f}%")

# Если не хватает до N, добавим ещё несколько
if len(available_names) < N:
    extra = [name for name in new_feature_names if name not in available_names][:N - len(available_names)]
    available_names.extend(extra)
    print(f"Добавили ещё {len(extra)} признаков для достижения N={N}")

print(f"\nИтоговый список отобранных признаков ({len(available_names)}):")
for name in available_names:
    print(f"  - {name}")

Выбрано признаков: 16 из 30
Доля: 53.3%

Итоговый список отобранных признаков (16):
  - num__ram
  - num__battery_power
  - num__px_height
  - num__px_width
  - num__sc_h
  - num__sc_w
  - num__mobile_wt
  - cat__four_g
  - cat__wifi
  - cat__touch_screen
  - cat__dual_sim
  - poly__ram
  - poly__battery_power
  - poly__ram battery_power
  - binner__mobile_wt
  - binner__sc_h


In [28]:
# Получаем индексы отобранных признаков в общем массиве
selected_indices = [list(new_feature_names).index(name) for name in available_names]

print(f"Индексы отобранных признаков: {selected_indices}")
print(f"Количество: {len(selected_indices)}")

# Сохраняем в два файла (по заданию)
with open('expert_selected_names.txt', 'w') as f:
    f.write('\n'.join(available_names))

with open('expert_selected_idx.txt', 'w') as f:
    f.write(str(selected_indices))



Индексы отобранных признаков: [9, 0, 7, 8, 10, 11, 5, 19, 23, 22, 18, 24, 25, 26, 27, 28]
Количество: 16


In [29]:
# Класс для выбора столбцов по индексам в pipeline
class ColumnExtractor:
    def __init__(self, cols):
        self.cols = cols
    
    def transform(self, X):
        return X[:, self.cols]
    
    def fit(self, X, y=None):
        return self



In [30]:
# Создаём pipeline: препроцессор (из пункта 10) → отбор столбцов → модель
expert_pipeline = Pipeline(steps=[
    ('preprocessor', fe_preprocessor),      # из пункта 10
    ('selector', ColumnExtractor(selected_indices)),  # отбор признаков
    ('model', RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42))
])



In [31]:
# Обучаем модель
expert_pipeline.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_pred_expert = expert_pipeline.predict(X_test)
y_pred_proba_expert = expert_pipeline.predict_proba(X_test)

# Метрики
expert_metrics = {
    'precision': precision_score(y_test, y_pred_expert, average='macro', zero_division=0),
    'recall': recall_score(y_test, y_pred_expert, average='macro', zero_division=0),
    'f1': f1_score(y_test, y_pred_expert, average='macro', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_expert, multi_class='ovr', average='macro')
}

print(" Метрики модели с экспертным отбором:")
for metric, value in expert_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\n Сравнение с предыдущими моделями:")
print(f"  Baseline f1:        {baseline_metrics['f1']:.4f}")
print(f"  FE sklearn f1:      {fe_metrics['f1']:.4f}")
print(f"  Expert selection f1: {expert_metrics['f1']:.4f}")

 Метрики модели с экспертным отбором:
  precision: 0.9020
  recall: 0.8993
  f1: 0.8993
  roc_auc: 0.9889

 Сравнение с предыдущими моделями:
  Baseline f1:        0.8517
  FE sklearn f1:      0.8889
  Expert selection f1: 0.8993


In [32]:
# Получаем ID эксперимента
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

RUN_NAME_EXPERT = "expert_feature_selection"

with mlflow.start_run(run_name=RUN_NAME_EXPERT, experiment_id=experiment_id) as run:
    expert_run_id = run.info.run_id
    
    # Логируем модель
    mlflow.sklearn.log_model(
        sk_model=expert_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        pip_requirements=['scikit-learn', 'pandas', 'numpy']
    )
    
    # Логируем метрики
    mlflow.log_metrics(expert_metrics)
    
    # Логируем параметры
    mlflow.log_params(expert_pipeline.get_params())
    
    # Логируем файлы с названиями и индексами отобранных столбцов
    mlflow.log_artifact('expert_selected_names.txt')
    mlflow.log_artifact('expert_selected_idx.txt')
    
    

2026/09/29 05:04:19 INFO mlflow.tracking._tracking_service.client: 🏃 View run expert_feature_selection at: http://127.0.0.1:5000/#/experiments/1/runs/6d53a88e6f3b49a19a45987c771cf98a.
2026/09/29 05:04:19 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


### Optuna


In [33]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Отключаем лишние предупреждения Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)



In [34]:
def objective(trial):
    """
    Функция цели для Optuna.
    Возвращает f1_macro — будем МАКСИМИЗИРОВАТЬ.
    """
    # Предлагаемые гиперпараметры
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    max_features = trial.suggest_float('max_features', 0.1, 1.0)
    
    # Создаём pipeline с теми же шагами, что и в expert_pipeline
    opt_pipeline = Pipeline(steps=[
        ('preprocessor', fe_preprocessor),
        ('selector', ColumnExtractor(selected_indices)),
        ('model', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Обучаем
    opt_pipeline.fit(X_train, y_train)
    
    # Предсказания на тесте
    preds = opt_pipeline.predict(X_test)
    
    # Метрика: f1_macro (для многоклассовой классификации)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    
    return f1



In [ ]:
# Создаём исследование с направлением МАКСИМИЗАЦИИ f1
study = optuna.create_study(
    study_name='rf_expert_selection_tuning',
    direction='maximize'  #  максимизируем f1
)

# Проводим 15 trials 
print(" Запуск Optuna... 15 trials")
study.optimize(objective, n_trials=15, show_progress_bar=True)



 Запуск Optuna... 15 trials


Best trial: 5. Best value: 0.910198: 100%|██████████| 15/15 [00:01<00:00,  7.65it/s]


In [37]:
# Получаем лучшие параметры
best_params = study.best_params

# Обучаем лучшую модель на train-выборке
opt_best_pipeline = Pipeline(steps=[
    ('preprocessor', fe_preprocessor),
    ('selector', ColumnExtractor(selected_indices)),
    ('model', RandomForestClassifier(
        **best_params,
        random_state=42,
        n_jobs=-1
    ))
])

opt_best_pipeline.fit(X_train, y_train)

# Предсказания и метрики на тесте
y_pred_opt = opt_best_pipeline.predict(X_test)
y_pred_proba_opt = opt_best_pipeline.predict_proba(X_test)

opt_metrics = {
    'f1': f1_score(y_test, y_pred_opt, average='macro', zero_division=0),
    'precision': precision_score(y_test, y_pred_opt, average='macro', zero_division=0),
    'recall': recall_score(y_test, y_pred_opt, average='macro', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_opt, multi_class='ovr', average='macro')
}

print(" Метрики лучшей Optuna-модели:")
for metric, value in opt_metrics.items():
    print(f"   {metric}: {value:.4f}")

print(f"\n Сравнение:")
print(f"   Baseline f1:          {baseline_metrics['f1']:.4f}")
print(f"   Expert selection f1:  {expert_metrics['f1']:.4f}")
print(f"   Optuna tuned f1:      {opt_metrics['f1']:.4f}")

 Метрики лучшей Optuna-модели:
   f1: 0.9102
   precision: 0.9104
   recall: 0.9104
   roc_auc: 0.9874

 Сравнение:
   Baseline f1:          0.8517
   Expert selection f1:  0.8993
   Optuna tuned f1:      0.9102


In [38]:
# Название эксперимента и запуска
RUN_NAME_OPTUNA = "optuna_rf_expert_selection"

# Получаем ID эксперимента
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME_OPTUNA, experiment_id=experiment_id) as run:
    opt_run_id = run.info.run_id
    
    # Логируем модель
    mlflow.sklearn.log_model(
        sk_model=opt_best_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        pip_requirements=['scikit-learn', 'pandas', 'numpy', 'optuna']
    )
    
    # Логируем метрики
    mlflow.log_metrics(opt_metrics)
    
    # Логируем лучшие гиперпараметры Optuna
    mlflow.log_params(best_params)
    mlflow.log_param('optimization_method', 'optuna')
    mlflow.log_param('n_trials', len(study.trials))
    mlflow.log_param('feature_selection', 'expert_eda')
    
    # Логируем файлы с отобранными признаками (для воспроизводимости)
    mlflow.log_artifact('expert_selected_names.txt')
    mlflow.log_artifact('expert_selected_idx.txt')
    
   

2026/09/29 05:12:59 INFO mlflow.tracking._tracking_service.client: 🏃 View run optuna_rf_expert_selection at: http://127.0.0.1:5000/#/experiments/1/runs/ee66377c01df4877ad7bd3682e9840d0.
2026/09/29 05:12:59 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [40]:
# Регистрируем как вторую версию модели
model_version_2 = mlflow.register_model(
    model_uri=f"runs:/{opt_run_id}/model",
    name=REGISTRY_MODEL_NAME
)


Registered model 'mobile_price_rf' already exists. Creating a new version of this model...
2026/09/29 05:13:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_rf, version 2
Created version '2' of model 'mobile_price_rf'.


### CatBoost

In [61]:
import subprocess

# Генерируем актуальный requirements.txt
result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)
requirements_content = result.stdout

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)



In [62]:
from catboost import CatBoostClassifier
import mlflow.catboost



In [63]:

catboost_pipeline = Pipeline(steps=[
    ('preprocessor', fe_preprocessor),
    ('selector', ColumnExtractor(selected_indices)),
    ('model', CatBoostClassifier(
        iterations=200,          # количество деревьев
        depth=6,                 # глубина деревьев
        learning_rate=0.1,       # скорость обучения
        random_seed=42,
        verbose=0                # отключаем вывод обучения
    ))
])


In [64]:
# Обучаем модель

catboost_pipeline.fit(X_train, y_train)


# Предсказания на тестовой выборке
y_pred_cb = catboost_pipeline.predict(X_test)
y_pred_proba_cb = catboost_pipeline.predict_proba(X_test)

# Метрики (те же, что и раньше — для классификации)
catboost_metrics = {
    'precision': precision_score(y_test, y_pred_cb, average='macro', zero_division=0),
    'recall': recall_score(y_test, y_pred_cb, average='macro', zero_division=0),
    'f1': f1_score(y_test, y_pred_cb, average='macro', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_cb, multi_class='ovr', average='macro')
}

print("\n CatBoost metrics:")
for metric, value in catboost_metrics.items():
    print(f"   {metric}: {value:.4f}")

print(f"\n Сравнение с предыдущими моделями:")
print(f"  Baseline f1:          {baseline_metrics['f1']:.4f}")
print(f"  FE sklearn f1:        {fe_metrics['f1']:.4f}")
print(f"  Expert selection f1:  {expert_metrics['f1']:.4f}")
print(f"  Optuna RF f1:         {opt_metrics['f1']:.4f}")
print(f"  CatBoost f1:          {catboost_metrics['f1']:.4f}")


 CatBoost metrics:
   precision: 0.9345
   recall: 0.9343
   f1: 0.9343
   roc_auc: 0.9955

 Сравнение с предыдущими моделями:
  Baseline f1:          0.8517
  FE sklearn f1:        0.8889
  Expert selection f1:  0.8993
  Optuna RF f1:         0.9102
  CatBoost f1:          0.9343


In [45]:
# Название запуска
RUN_NAME_CATBOOST = "catboost_expert_selection"

# Получаем ID эксперимента
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    catboost_run_id = run.info.run_id
    
    
    catboost_model = catboost_pipeline.named_steps['model']
    
    mlflow.catboost.log_model(
        cb_model=catboost_model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        pip_requirements=['scikit-learn', 'pandas', 'numpy', 'catboost']
    )
    
    # Логируем метрики
    mlflow.log_metrics(catboost_metrics)
    
    # Логируем параметры CatBoost
    mlflow.log_params(catboost_model.get_params())
    mlflow.log_param('feature_selection', 'expert_eda')
    mlflow.log_param('n_selected_features', len(selected_indices))
    
    # Логируем файлы с отобранными признаками
    mlflow.log_artifact('expert_selected_names.txt')
    mlflow.log_artifact('expert_selected_idx.txt')
    
   

2026/09/29 05:21:46 INFO mlflow.tracking._tracking_service.client: 🏃 View run catboost_expert_selection at: http://127.0.0.1:5000/#/experiments/1/runs/339fa3e1a0bb41d881b37f37d3aeb33a.
2026/09/29 05:21:46 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [65]:
# Регистрируем как третью версию модели
model_version_3 = mlflow.register_model(
    model_uri=f"runs:/{catboost_run_id}/model",
    name=REGISTRY_MODEL_NAME
)



Registered model 'mobile_price_rf' already exists. Creating a new version of this model...
2026/09/29 06:07:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_rf, version 4
Created version '4' of model 'mobile_price_rf'.


### Production

In [54]:
# Параметры CatBoost, которые мы использовали в пункте 15 (не из Optuna!)
catboost_params = {
    'iterations': 200,
    'depth': 6,
    'learning_rate': 0.1,
    'random_seed': 42,
    'verbose': 0
}



In [56]:
from catboost import CatBoostClassifier

# Создаём финальный pipeline
final_pipeline = Pipeline(steps=[
    ('preprocessor', fe_preprocessor),
    ('selector', ColumnExtractor(selected_indices)),
    ('model', CatBoostClassifier(**catboost_params))
])


final_pipeline.fit(X, y)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['battery_power',
                                                   'clock_speed', 'fc',
                                                   'int_memory', 'm_dep',
                                                   'mobile_wt', 'pc',
                                                   'px_height', 'px_width',
                                                   'ram', 'sc_h', 'sc_w',
                                                   'talk_time', 'total_pixels',
                                                   'pixel_density',
                                                   'battery_efficiency',
                                                   'total_camera']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown...
                                                                   PolynomialFeatures(include_bias=False,
                                                                                      interaction_only=True)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['ram', 'battery_power']),
                                                 ('binner',
                                                  KBinsDiscretizer(encode='ordinal',
                                                                   n_bins=3),
                                                  ['mobile_wt', 'sc_h',
                                                   'talk_time'])])),
                ('selector', <__main__.ColumnExtractor object at 0x31e8fa8a0>),
                ('model',
                 CatBoostClassifier(depth=6, iterations=200, learning_rate=0.1, random_seed=42, verbose=0))])

In [58]:
from mlflow.models import infer_signature


selected_names = available_names 

# Пример входных данных — первые 5 строк ПОЛНОЙ выборки
input_example_prod = X.head(5)

# Сигнатура модели
signature_prod = infer_signature(
    model_input=input_example_prod,
    model_output=final_pipeline.predict(input_example_prod)
)

# Сохраняем список используемых столбцов
with open('production_columns.txt', 'w') as f:
    f.write("=== Список признаков для Production-модели ===\n")
    f.write(f"Количество: {len(selected_names)}\n\n")
    f.write("Названия:\n")
    for i, name in enumerate(selected_names, 1):
        f.write(f"  {i}. {name}\n")
    f.write(f"\nИндексы: {selected_indices}\n")



/Users/komarevtsev/Documents/project_Komarevtsev/.venv_project_Komarevtsev/lib/python3.12/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [66]:
RUN_NAME_PROD = "production_catboost_final"

# Получаем ID эксперимента
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME_PROD, experiment_id=experiment_id) as run:
    prod_run_id = run.info.run_id
   
    catboost_model = final_pipeline.named_steps['model']
    
    mlflow.catboost.log_model(
        cb_model=catboost_model,
        artifact_path="model",
        signature=signature_prod,
        input_example=input_example_prod,
        pip_requirements=['scikit-learn', 'pandas', 'numpy', 'catboost']
    )
    
    # Логируем параметры модели
    mlflow.log_params(catboost_model.get_params())
    mlflow.log_param('feature_selection', 'expert_eda')
    mlflow.log_param('n_selected_features', len(selected_indices))
    mlflow.log_param('trained_on', 'full_dataset')
    mlflow.log_param('optimization_method', 'manual_params_from_p15')
    
    # Логируем артефакты
    mlflow.log_artifact('production_columns.txt')
    mlflow.log_artifact('requirements.txt')
    
    

2026/09/29 06:08:47 INFO mlflow.tracking._tracking_service.client: 🏃 View run production_catboost_final at: http://127.0.0.1:5000/#/experiments/1/runs/9909675381a54e02ac66ab420f252038.
2026/09/29 06:08:47 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [67]:
# Регистрируем модель в реестре 
model_version_prod = mlflow.register_model(
    model_uri=f"runs:/{prod_run_id}/model",
    name=REGISTRY_MODEL_NAME
)


client = mlflow.tracking.MlflowClient()

client.transition_model_version_stage(
    name=REGISTRY_MODEL_NAME,
    version=model_version_prod.version,
    stage="Production"
)



Registered model 'mobile_price_rf' already exists. Creating a new version of this model...
2026/09/29 06:08:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_rf, version 5
Created version '5' of model 'mobile_price_rf'.
/var/folders/b5/hp3_ng3910jc3m_wqhf_gz8w0000gn/T/ipykernel_13621/261414011.py:10: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1790651337141, current_stage='Production', description='', last_updated_timestamp=1790651337148, name='mobile_price_rf', run_id='9909675381a54e02ac66ab420f252038', run_link='', source='/Users/komarevtsev/Documents/project_Komarevtsev/mlflow/mlartifacts/1/9909675381a54e02ac66ab420f252038/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='5'>